# 04 Preprocessing — Women's

This notebook constructs matchup-level features for all datasets (training, Stage 1 grid, Stage 2 grid). The dominant pattern from research is **difference features**: `TeamA_stat - TeamB_stat` for each matchup.

**Key difference from men's: No Massey Ordinals are available for women's basketball.** Feature tiers are adjusted accordingly.

**Feature tiers (from EDA and research):**
1. Seed difference
2. Efficiency differences (OffEff, DefEff, NetEff)
3. Win percentage and point differential differences
4. Shooting and box score stat differences
5. Conference strength (power conference flag)

**Training data augmentation**: "Flip and double" — each training matchup appears twice with features and label negated/flipped, preventing the model from learning ordering artifacts.

**Inputs**:
- From `03_data_split/womens/`: `tourney_matchups.parquet`, `prediction_grid_stage1.parquet`, `prediction_grid_stage2.parquet`
- From `01_data_joining/womens/`: `team_season_stats.parquet`, `tourney_seeds.parquet`, `team_metadata.parquet`

**Outputs** (to S3 `04_preprocessing/womens/`):
1. `train_features.parquet` — training matchups with features, labels, and fold assignments (flip-doubled)
2. `stage1_features.parquet` — Stage 1 grid with features and labels where available
3. `stage2_features.parquet` — Stage 2 grid with features (2026 predictions)
4. `feature_columns.parquet` — list of feature column names for modeling

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

#### Functions

In [2]:
def read_parquet_joining(filename):
    """Read parquet from 01_data_joining on S3 or local."""
    try:
        return pd.read_parquet(f"s3://{BUCKET}/01_data_joining/{GENDER}/{filename}")
    except Exception:
        return pd.read_parquet(f"../01_data_joining/output/{filename}")

def read_parquet_split(filename):
    """Read parquet from 03_data_split on S3 or local."""
    try:
        return pd.read_parquet(f"s3://{BUCKET}/03_data_split/{GENDER}/{filename}")
    except Exception:
        return pd.read_parquet(f"../03_data_split/output/{filename}")

In [3]:
def build_matchup_features(matchup_df, team_stats, seeds, team_meta):
    """
    For each matchup (Season, TeamA, TeamB), compute difference features.
    TeamA is always the lower-ID team (submission format).
    Features are computed as TeamA_stat - TeamB_stat.
    
    NOTE: No massey parameter — Massey Ordinals are not available for women's basketball.
    """
    df = matchup_df.copy()
    
    # --- Seed features ---
    seed_cols = seeds[['Season', 'TeamID', 'SeedNum']]
    df = df.merge(seed_cols, left_on=['Season', 'TeamA'], right_on=['Season', 'TeamID'], how='left')
    df = df.rename(columns={'SeedNum': 'SeedA'}).drop(columns=['TeamID'], errors='ignore')
    df = df.merge(seed_cols, left_on=['Season', 'TeamB'], right_on=['Season', 'TeamID'], how='left')
    df = df.rename(columns={'SeedNum': 'SeedB'}).drop(columns=['TeamID'], errors='ignore')
    df['SeedDiff'] = df['SeedA'] - df['SeedB']
    
    # --- Team season stat features ---
    stat_cols = ['WinPct', 'AvgPointDiff', 'OffEff', 'DefEff', 'NetEff',
                 'FGPct', 'FG3Pct', 'FTPct',
                 'AvgTO', 'AvgStl', 'AvgBlk', 'AvgOR', 'AvgDR', 'AvgAst',
                 'OppFGPct', 'OppFG3Pct', 'AvgPoss']
    # Only use columns that exist
    stat_cols = [c for c in stat_cols if c in team_stats.columns]
    
    stats_subset = team_stats[['Season', 'TeamID'] + stat_cols]
    
    df = df.merge(stats_subset, left_on=['Season', 'TeamA'], right_on=['Season', 'TeamID'], how='left')
    df = df.rename(columns={c: f'{c}_A' for c in stat_cols}).drop(columns=['TeamID'], errors='ignore')
    df = df.merge(stats_subset, left_on=['Season', 'TeamB'], right_on=['Season', 'TeamID'], how='left')
    df = df.rename(columns={c: f'{c}_B' for c in stat_cols}).drop(columns=['TeamID'], errors='ignore')
    
    for col in stat_cols:
        df[f'{col}Diff'] = df[f'{col}_A'] - df[f'{col}_B']
    
    # --- Conference / power conference features ---
    meta_subset = team_meta[['Season', 'TeamID', 'IsPowerConf']]
    df = df.merge(meta_subset, left_on=['Season', 'TeamA'], right_on=['Season', 'TeamID'], how='left')
    df = df.rename(columns={'IsPowerConf': 'IsPowerConfA'}).drop(columns=['TeamID'], errors='ignore')
    df = df.merge(meta_subset, left_on=['Season', 'TeamB'], right_on=['Season', 'TeamID'], how='left')
    df = df.rename(columns={'IsPowerConf': 'IsPowerConfB'}).drop(columns=['TeamID'], errors='ignore')
    df['IsPowerConfDiff'] = df['IsPowerConfA'] - df['IsPowerConfB']
    
    return df

In [4]:
def get_feature_columns(df):
    """
    Return list of feature columns (all Diff columns plus seed values).
    Excludes ID, Season, TeamA, TeamB, Label, Fold, IsStage1Val, and raw per-team columns.
    """
    diff_cols = [c for c in df.columns if c.endswith('Diff')]
    extra_features = ['SeedA', 'SeedB']
    extra_features = [c for c in extra_features if c in df.columns]
    return diff_cols + extra_features

In [5]:
def flip_and_double(df, feature_cols, label_col='Label'):
    """
    Augment training data by adding mirror of each matchup.
    For the flipped copy: negate all difference features and flip the label.
    This prevents the model from learning artifacts based on team ID ordering.
    """
    original = df.copy()
    flipped = df.copy()
    
    # Negate all difference features
    diff_cols = [c for c in feature_cols if c.endswith('Diff')]
    for col in diff_cols:
        flipped[col] = -flipped[col]
    
    # Swap SeedA/SeedB if present
    if 'SeedA' in flipped.columns and 'SeedB' in flipped.columns:
        flipped['SeedA'], flipped['SeedB'] = flipped['SeedB'].copy(), flipped['SeedA'].copy()
    
    # Flip label
    if label_col in flipped.columns:
        flipped[label_col] = 1 - flipped[label_col]
    
    # Swap TeamA/TeamB for identification
    flipped['TeamA'], flipped['TeamB'] = flipped['TeamB'].copy(), flipped['TeamA'].copy()
    
    return pd.concat([original, flipped], ignore_index=True)

#### Constants

In [6]:
BUCKET = "march-machine-learning-mania-2026"
GENDER = "womens"
STAGE = "04_preprocessing"
OUTPUT_PREFIX = f"s3://{BUCKET}/{STAGE}/{GENDER}/"

LOCAL_OUTPUT = "output/"

#### Make output directory

In [7]:
os.makedirs(LOCAL_OUTPUT, exist_ok=True)

## 1. Load Data

Note: Unlike the men's pipeline, there are no Massey Ordinals for women's basketball. The feature set relies on seeds, team season stats, and conference metadata.

In [8]:
# From 03_data_split
tourney_matchups = read_parquet_split("tourney_matchups.parquet")
grid_stage1 = read_parquet_split("prediction_grid_stage1.parquet")
grid_stage2 = read_parquet_split("prediction_grid_stage2.parquet")

# From 01_data_joining (NO massey ordinals for women's)
team_stats = read_parquet_joining("team_season_stats.parquet")
seeds = read_parquet_joining("tourney_seeds.parquet")
team_meta = read_parquet_joining("team_metadata.parquet")

print(f"Tournament matchups: {tourney_matchups.shape}")
print(f"Stage 1 grid: {grid_stage1.shape}")
print(f"Stage 2 grid: {grid_stage2.shape}")
print(f"Team stats: {team_stats.shape}")
print(f"Seeds: {seeds.shape}")
print(f"Team metadata: {team_meta.shape}")

Tournament matchups: (1717, 7)
Stage 1 grid: (258131, 5)
Stage 2 grid: (65703, 4)
Team stats: (9851, 45)
Seeds: (1744, 6)
Team metadata: (9853, 6)


## 2. Build Features for Training Data

In [9]:
train_feat = build_matchup_features(tourney_matchups, team_stats, seeds, team_meta)

feature_cols = get_feature_columns(train_feat)
print(f"Training features built: {train_feat.shape}")
print(f"Feature columns ({len(feature_cols)}): {feature_cols}")

# Check missing values in features
missing = train_feat[feature_cols].isnull().mean().sort_values(ascending=False)
print(f"\nFeature missing rates:")
for feat, rate in missing.items():
    if rate > 0:
        print(f"  {feat}: {rate:.3f}")
if (missing == 0).all():
    print("  No missing values!")

Training features built: (1717, 64)
Feature columns (21): ['SeedDiff', 'WinPctDiff', 'AvgPointDiffDiff', 'OffEffDiff', 'DefEffDiff', 'NetEffDiff', 'FGPctDiff', 'FG3PctDiff', 'FTPctDiff', 'AvgTODiff', 'AvgStlDiff', 'AvgBlkDiff', 'AvgORDiff', 'AvgDRDiff', 'AvgAstDiff', 'OppFGPctDiff', 'OppFG3PctDiff', 'AvgPossDiff', 'IsPowerConfDiff', 'SeedA', 'SeedB']

Feature missing rates:
  DefEffDiff: 0.440
  OffEffDiff: 0.440
  FG3PctDiff: 0.440
  FGPctDiff: 0.440
  NetEffDiff: 0.440
  AvgDRDiff: 0.440
  AvgAstDiff: 0.440
  OppFGPctDiff: 0.440
  FTPctDiff: 0.440
  AvgTODiff: 0.440
  AvgStlDiff: 0.440
  AvgBlkDiff: 0.440
  AvgORDiff: 0.440
  AvgPossDiff: 0.440
  OppFG3PctDiff: 0.440


## 3. Flip and Double Training Data

Each matchup produces two rows: the original and its mirror (features negated, label flipped). This ensures the model cannot learn from the arbitrary lower-ID-first ordering.

In [10]:
train_augmented = flip_and_double(train_feat, feature_cols)

print(f"Before flip-and-double: {train_feat.shape}")
print(f"After flip-and-double: {train_augmented.shape}")
print(f"Label balance after: {train_augmented.Label.mean():.3f} (should be ~0.5)")

Before flip-and-double: (1717, 64)
After flip-and-double: (3434, 64)
Label balance after: 0.500 (should be ~0.5)


## 4. Build Features for Prediction Grids

Same feature construction for Stage 1 and Stage 2 grids. No flip-and-double needed — predictions are made in the canonical (lower-ID first) format.

In [11]:
stage1_feat = build_matchup_features(grid_stage1, team_stats, seeds, team_meta)
print(f"Stage 1 features: {stage1_feat.shape}")

# Check feature coverage
s1_missing = stage1_feat[feature_cols].isnull().mean()
print(f"Stage 1 feature missing rates (top 5):")
for feat, rate in s1_missing.sort_values(ascending=False).head().items():
    print(f"  {feat}: {rate:.3f}")

Stage 1 features: (258131, 62)
Stage 1 feature missing rates (top 5):
  SeedDiff: 0.965
  SeedB: 0.815
  SeedA: 0.807
  WinPctDiff: 0.000
  AvgPointDiffDiff: 0.000


In [12]:
stage2_feat = build_matchup_features(grid_stage2, team_stats, seeds, team_meta)
print(f"Stage 2 features: {stage2_feat.shape}")

# Check feature coverage
s2_missing = stage2_feat[feature_cols].isnull().mean()
print(f"Stage 2 feature missing rates (top 5):")
for feat, rate in s2_missing.sort_values(ascending=False).head().items():
    print(f"  {feat}: {rate:.3f}")

Stage 2 features: (65703, 61)
Stage 2 feature missing rates (top 5):
  SeedDiff: 1.000
  SeedA: 1.000
  SeedB: 1.000
  WinPctDiff: 0.000
  AvgPointDiffDiff: 0.000


## 5. Feature Sanity Checks

In [13]:
# Verify difference features are antisymmetric in training data
# After flip-and-double, the mean of every Diff feature should be ~0
diff_cols = [c for c in feature_cols if c.endswith('Diff')]
diff_means = train_augmented[diff_cols].mean()

print("Diff feature means after flip-and-double (should all be ~0):")
for feat, mean_val in diff_means.items():
    status = "OK" if abs(mean_val) < 0.01 else "CHECK"
    print(f"  {feat}: {mean_val:+.6f} [{status}]")

Diff feature means after flip-and-double (should all be ~0):
  SeedDiff: +0.000000 [OK]
  WinPctDiff: +0.000000 [OK]
  AvgPointDiffDiff: +0.000000 [OK]
  OffEffDiff: +0.000000 [OK]
  DefEffDiff: +0.000000 [OK]
  NetEffDiff: +0.000000 [OK]
  FGPctDiff: -0.000000 [OK]
  FG3PctDiff: -0.000000 [OK]
  FTPctDiff: +0.000000 [OK]
  AvgTODiff: -0.000000 [OK]
  AvgStlDiff: -0.000000 [OK]
  AvgBlkDiff: +0.000000 [OK]
  AvgORDiff: +0.000000 [OK]
  AvgDRDiff: -0.000000 [OK]
  AvgAstDiff: -0.000000 [OK]
  OppFGPctDiff: +0.000000 [OK]
  OppFG3PctDiff: +0.000000 [OK]
  AvgPossDiff: -0.000000 [OK]
  IsPowerConfDiff: +0.000000 [OK]


In [14]:
# Quick check: do features correlate with labels in expected direction?
# SeedDiff should be negative (lower seed = better = more likely to win)
# WinPctDiff should be positive (higher win% = more likely to win)
# NetEffDiff should be positive (higher net efficiency = more likely to win)
key_features = ['SeedDiff', 'WinPctDiff', 'NetEffDiff', 'AvgPointDiffDiff']
key_features = [f for f in key_features if f in train_feat.columns]

print("Feature-Label correlations (before flip-and-double):")
for feat in key_features:
    corr = train_feat[[feat, 'Label']].dropna().corr().iloc[0, 1]
    print(f"  {feat}: {corr:+.3f}")

Feature-Label correlations (before flip-and-double):
  SeedDiff: -0.624
  WinPctDiff: +0.395
  NetEffDiff: +0.476
  AvgPointDiffDiff: +0.490


## 6. Save Outputs

In [15]:
# Save feature column names for downstream modeling
feature_cols_df = pd.DataFrame({'feature': feature_cols})

outputs = {
    'train_features': train_augmented,
    'stage1_features': stage1_feat,
    'stage2_features': stage2_feat,
    'feature_columns': feature_cols_df,
}

for name, df in outputs.items():
    try:
        s3_path = f"{OUTPUT_PREFIX}{name}.parquet"
        df.to_parquet(s3_path, index=False)
        print(f"Saved to S3: {s3_path} ({df.shape})")
    except Exception as e:
        print(f"S3 save failed for {name}: {e}")

Saved to S3: s3://march-machine-learning-mania-2026/04_preprocessing/womens/train_features.parquet ((3434, 64))
Saved to S3: s3://march-machine-learning-mania-2026/04_preprocessing/womens/stage1_features.parquet ((258131, 62))
Saved to S3: s3://march-machine-learning-mania-2026/04_preprocessing/womens/stage2_features.parquet ((65703, 61))
Saved to S3: s3://march-machine-learning-mania-2026/04_preprocessing/womens/feature_columns.parquet ((21, 1))


## 7. Output Summary

In [16]:
print("=" * 60)
print("PREPROCESSING SUMMARY — WOMEN'S")
print("=" * 60)

print(f"\ntrain_features:")
print(f"  Shape: {train_augmented.shape} (flip-doubled from {train_feat.shape[0]} matchups)")
print(f"  Label balance: {train_augmented.Label.mean():.3f}")
print(f"  Seasons: {train_augmented.Season.min()} - {train_augmented.Season.max()}")

print(f"\nstage1_features:")
print(f"  Shape: {stage1_feat.shape}")
print(f"  Rows with labels: {stage1_feat.Label.notna().sum() if 'Label' in stage1_feat.columns else 'N/A'}")

print(f"\nstage2_features:")
print(f"  Shape: {stage2_feat.shape}")

print(f"\nFeature columns ({len(feature_cols)}):")
for f in feature_cols:
    print(f"  - {f}")

print(f"\nNote: No Massey Ordinal features (not available for women's basketball)")

PREPROCESSING SUMMARY — WOMEN'S

train_features:
  Shape: (3434, 64) (flip-doubled from 1717 matchups)
  Label balance: 0.500
  Seasons: 1998 - 2025

stage1_features:
  Shape: (258131, 62)
  Rows with labels: 268

stage2_features:
  Shape: (65703, 61)

Feature columns (21):
  - SeedDiff
  - WinPctDiff
  - AvgPointDiffDiff
  - OffEffDiff
  - DefEffDiff
  - NetEffDiff
  - FGPctDiff
  - FG3PctDiff
  - FTPctDiff
  - AvgTODiff
  - AvgStlDiff
  - AvgBlkDiff
  - AvgORDiff
  - AvgDRDiff
  - AvgAstDiff
  - OppFGPctDiff
  - OppFG3PctDiff
  - AvgPossDiff
  - IsPowerConfDiff
  - SeedA
  - SeedB

Note: No Massey Ordinal features (not available for women's basketball)
